# General setup

In [1]:
import pandas as pd
import gc
import os
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import userdata

from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

print("Default GPU Device:", tf.test.gpu_device_name() if tf.test.gpu_device_name() else "No GPU available")


Default GPU Device: /device:GPU:0


# Environment setup

In [2]:
# Retrieve secrets from the Colab "Secrets" tab
try:
    KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
    KAGGLE_KEY = userdata.get('KAGGLE_KEY')
except userdata.SecretNotFoundError as e:
    print(f"Error: {e}")
    print("Make sure you've added these keys in the Secrets tab (the key icon on the left).")

In [3]:
# Google Drive setup

from google.colab import drive
drive.mount('/content/drive')
files_base = '/content/drive/MyDrive/AI ML Certificate with Berkeley Haas/capstone/lendingclub'

Mounted at /content/drive


# Fetching Datasets

## Kaggle setup
This section does general setup for Kaggle that will be used for the LendingClub dataset as well as the S&P 500 dataset.

In [4]:
!pip install kaggle

In [5]:
# Get Kaggle username and API token, and make them available to the API
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

download_path = os.path.join(files_base, 'kaggle')

## Fetching the LendingClub dataset from Kaggle
This section pertains specifically to the LendingClub dataset.

In [6]:
skip_download_lendingclub = True # TODO: set to false
if not skip_download_lendingclub:
  from kaggle.api.kaggle_api_extended import KaggleApi
  dataset = 'wordsforthewise/lending-club'

  # Initialize API
  api = KaggleApi()
  api.authenticate()
  print('Authenticated to Kaggle')

  # Download dataset
  print(f'Downloading dataset {dataset} to {download_path}')
  api.dataset_download_files(dataset, path=download_path, unzip=True)
  print('Downloaded files')

In [7]:
# File paths (note subtle case differences)
accepted_filename = os.path.join(
    'accepted_2007_to_2018q4.csv', 'accepted_2007_to_2018Q4.csv')
rejected_filename = os.path.join(
    'rejected_2007_to_2018q4.csv', 'rejected_2007_to_2018Q4.csv')
# gzipped input files are extracted into subfolders
accepted_path = os.path.join(download_path, accepted_filename)
rejected_path = os.path.join(download_path, rejected_filename)

In [8]:
# Load into Pandas
accepted_df = pd.read_csv(accepted_path, low_memory=False)
print("Accepted shape:", accepted_df.shape)

# We won't be using the rejected loans to make decsions, because they lack
# actual performance data. However, to access this loan data, uncomment the
# lines below.
#rejected_df = pd.read_csv(rejected_path, low_memory=True)
#print("Rejected shape:", rejected_df.shape)

Accepted shape: (2260701, 151)


## Fetching the Federal Funds Effective rate dataset from the St. Louis Fed
The FEDFUNDS dataset is [available from the St. Louis Fed](https://fred.stlouisfed.org/series/FEDFUNDS). It is a simple dataset, providing a date and the percentage rate.

In [9]:
# Fetch a dataset from 2005-Jan through 2020-Jan,
# overlapping with the LendingClub dataset.
url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?bgcolor=%23ebf3fb&chart_type=line&drp=0&fo=open%20sans&graph_bgcolor=%23ffffff&height=450&mode=fred&recession_bars=on&txtcolor=%23444444&ts=12&tts=12&width=1320&nt=0&thu=0&trc=0&show_legend=yes&show_axis_titles=yes&show_tooltip=yes&id=FEDFUNDS&scale=left&cosd=2005-01-01&coed=2020-01-01&line_color=%230073e6&link_values=false&line_style=solid&mark_type=none&mw=3&lw=3&ost=-99999&oet=99999&mma=0&fml=a&fq=Monthly&fam=avg&fgst=lin&fgsnd=2020-02-01&line_index=1&transformation=lin&vintage_date=2026-02-22&revision_date=2026-02-22&nd=1954-07-01'
fed_funds_df = pd.read_csv(url, parse_dates=['observation_date'])

## Fetching the S&P 500 dataset from Kaggle

In [10]:
skip_download_sp500 = False
if not skip_download_sp500:
  from kaggle.api.kaggle_api_extended import KaggleApi
  dataset = 'henryhan117/sp-500-historical-data'

  # Initialize API
  api = KaggleApi()
  api.authenticate()
  print('Authenticated to Kaggle')

  # Download dataset
  print(f'Downloading dataset {dataset} to {download_path}')
  api.dataset_download_files(dataset, path=download_path, unzip=True)
  print('Downloaded files')

Authenticated to Kaggle
Dataset URL: https://www.kaggle.com/datasets/henryhan117/sp-500-historical-data
Downloaded files


In [11]:
sp500_filename = 'SPX.csv'
sp500_path = os.path.join(download_path, sp500_filename)

sp500_df = pd.read_csv(sp500_path, parse_dates=["Date"])
print("S&P 500 shape: ", sp500_df.shape)

S&P 500 shape:  (23323, 7)


# Exploring the data

A complete analysis of the input datasets can be found in the notebook `lendingclub_eda.ipynb`.

Here we retain only the portion of code from that analysis necessary for the final overall analysis. It is intentionally minimalistic; for grading purposes, please review the EDA notebook for a more complete description of the same code.

## LendingClub
This section is an exploration of the LendingClub dataset.

In [12]:
# Don't truncate these columns, even though there's a lot of them.
pd.set_option('display.max_rows', 500)       # show all rows
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.width', 80)            # let it use full console width

In [13]:
# Figure out how many NaNs we have to deal with, since we only want to retain
# columns with >20% valid data.
nans_percentage = accepted_df.isna().mean() * 100
mostly_valid_columns = nans_percentage[nans_percentage > 20]

In [14]:
# Drop columns that are 20%+ NaN. The data is too messy to use; and, as it turns
# out, these columns weren't useful anyway since they generally pertain to
# information only known during the duration of the loan, so they can't be used
# at lending decision time.
nan_columns_to_drop = accepted_df.columns[nans_percentage > 20]
print(nan_columns_to_drop)
accepted_df.drop(columns=nan_columns_to_drop, inplace=True)

# Now, drop rows that have a NaN anywhre in them
row_has_nan = accepted_df.isna().any(axis=1)
percent_rows_with_nan = row_has_nan.mean() * 100

Index(['member_id', 'desc', 'mths_since_last_delinq', 'mths_since_last_record',
       'next_pymnt_d', 'mths_since_last_major_derog', 'annual_inc_joint',
       'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_act_il',
       'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il',
       'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util',
       'inq_fi', 'total_cu_tl', 'inq_last_12m', 'mths_since_recent_bc_dlq',
       'mths_since_recent_revol_delinq', 'revol_bal_joint',
       'sec_app_fico_range_low', 'sec_app_fico_range_high',
       'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths',
       'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util',
       'sec_app_open_act_il', 'sec_app_num_rev_accts',
       'sec_app_chargeoff_within_12_mths',
       'sec_app_collections_12_mths_ex_med',
       'sec_app_mths_since_last_major_derog', 'hardship_type',
       'hardship_reason', 'hardship_status', 'deferral_term',
       'hardship_amount'

In [15]:
accepted_df = accepted_df.copy()[['grade', 'sub_grade', 'issue_d', 'loan_status', 'loan_amnt']].dropna()

In [16]:
letter_order = ['A', 'B', 'C', 'D', 'E', 'F']
letter_rank = {letter: i for i, letter in enumerate(letter_order)}

letters = accepted_df['sub_grade'].str[0]
numbers = accepted_df['sub_grade'].str[1].astype(int)
numbers = numbers.astype(int)

accepted_df['grade_score'] = (
    (5 - letters.map(letter_rank)) * 5 + (6 - numbers)
)

In [17]:
# Standardize a function to convert datetime to a numerical index; we'll use the
# same system for this and for economic data. (We assume all years are leap
# years and all months have 31 days.)
def datetime_to_index(dt):
  return dt.year*366 + dt.month*31 + dt.day

accepted_df['issue_d_dt'] = pd.to_datetime(
    accepted_df['issue_d'], format='%b-%Y')
accepted_df['issue_d_month_index'] = datetime_to_index(
    accepted_df['issue_d_dt'].dt)

print(accepted_df[['issue_d', 'issue_d_dt', 'issue_d_month_index']].sample(10))

          issue_d issue_d_dt  issue_d_month_index
460443   Feb-2018 2018-02-01               738651
1531989  May-2018 2018-05-01               738744
187844   Aug-2015 2015-08-01               737739
139693   Sep-2015 2015-09-01               737770
1160649  Oct-2014 2014-10-01               737435
393210   Jan-2015 2015-01-01               737522
181728   Aug-2015 2015-08-01               737739
916133   Jun-2017 2017-06-01               738409
635374   Jul-2017 2017-07-01               738440
444330   Mar-2018 2018-03-01               738682


In [18]:
# Let's ignore those very few non-compliant loans; not sure how to make heads or tails of those.
accepted_df = accepted_df[~accepted_df['loan_status'].isin([
    'Does not meet the credit policy. Status:Fully Paid',
    'Does not meet the credit policy. Status:Charged Off'])
]

In [19]:
loan_status_mapping = {
    "Fully Paid":         7,
    "Current":            6,

    "In Grace Period":    5,
    "Late (16-30 days)":  4,
    "Late (31-120 days)": 3,

    "Charged Off":        2,
    "Default" :           1
}

accepted_df['loan_status_code'] = accepted_df['loan_status'].dropna().map(loan_status_mapping).astype(int)

## Fed Funds Effective Rate

In [20]:
fed_funds_df['date_index'] = datetime_to_index(
    fed_funds_df['observation_date'].dt)

In [21]:
fed_funds_df['dFEDFUNDS'] = fed_funds_df['FEDFUNDS'].diff()

## S&P 500

In [22]:
sp500_df['date_index'] = datetime_to_index(
    sp500_df['Date'].dt)

In [23]:
yrs_to_average = 2
sp500_df['avg_close'] = sp500_df['Close'].rolling(yrs_to_average * 5*52).mean()
sp500_df['close_above_avg'] = sp500_df['Close'] - sp500_df['avg_close']

# Data Modeling

From here, we'll start with making some predictions.

## LendingClub modeling by grade

At a base level, we'll assume that LendingClub has done a pretty good job of choosing borrower grades as a function of all available data; from here, we can try to predict loan outcomes as a function of that initial grade. We'll do that with a neural net.

To start with, let's simply try to predict loan status codes (which describe the final loan outcome) with borrower grade.

In [81]:
# For training speed, use only a subset of the data.
# When using _all_ data for training, set data_frac = 1.0.
data_frac = 1.0
epochs = 7

In [ ]:
df2 = accepted_df[['grade_score', 'loan_status_code']].dropna()
df2 = df2.sample(frac=data_frac)

t = make_column_transformer((StandardScaler(), ['grade_score']),
                            remainder='passthrough').set_output(transform="pandas")
df2_scaled = t.fit_transform(df2)

X = df2_scaled[['standardscaler__grade_score']]
y = to_categorical(df2['loan_status_code'].astype(int))

(X_train, X_test, y_train, y_test) = train_test_split(X, y)

model = Sequential([
    Dense(31, activation='relu'),
    Dense(8, activation='softmax')
])
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
history2 = model.fit(X_train, y_train, epochs=epochs, validation_data=(X_test, y_test))

# Plot the results over time
plt.plot(history2.history['loss'])
plt.plot(history2.history['val_loss'])
plt.legend(['Training loss', 'Validation loss'])

plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')

Epoch 1/7
52640/52640 ━━━━━━━━━━━━━━━━━━━━ 159s 3ms/step - accuracy: 0.4770 - loss: 1.0253 - val_accuracy: 0.4784 - val_loss: 1.0242
Epoch 2/7
52640/52640 ━━━━━━━━━━━━━━━━━━━━ 157s 3ms/step - accuracy: 0.4771 - loss: 1.0231 - val_accuracy: 0.4779 - val_loss: 1.0237
Epoch 3/7
52640/52640 ━━━━━━━━━━━━━━━━━━━━ 157s 3ms/step - accuracy: 0.4772 - loss: 1.0232 - val_accuracy: 0.4779 - val_loss: 1.0236
Epoch 4/7
52640/52640 ━━━━━━━━━━━━━━━━━━━━ 156s 3ms/step - accuracy: 0.4771 - loss: 1.0233 - val_accuracy: 0.4779 - val_loss: 1.0231
Epoch 5/7
 6243/52640 ━━━━━━━━━━━━━━━━━━━━ 1:44 2ms/step - accuracy: 0.4780 - loss: 1.0202

That's a great start—but wait! We've used the standard `train_test_split` function to divide the data. Since data will tend to have trends over time, this may be a bit too generous to the testing model; that is, we may be experiencing data leakage.

So instead, let's split based on a dividing line in time. We'll train before that time, and test afterward. This is much more like realistic data as it would be used for actual loan decisions: Train a model based on the best data up to a date, then launch that model and see how it performs over time—_without_ the benefit of being able to "see the future."

Here we'll redo the analysis.

In [ ]:
df2 = accepted_df[['grade_score', 'loan_status_code', 'issue_d_month_index']].dropna()
df2 = df2.sample(frac=data_frac)

t = make_column_transformer((StandardScaler(), ['grade_score']),
                            remainder='passthrough').set_output(transform="pandas")
df2_scaled = t.fit_transform(df2)

# Split based on time
test_size = 0.25 # as in the train_test_split parameter
cutoff_index = df2['issue_d_month_index'].quantile(1 - test_size)

df2_scaled_train = df2_scaled[df2['issue_d_month_index'] <= cutoff_index]
df2_scaled_test = df2_scaled[df2['issue_d_month_index'] > cutoff_index]

# Define X and y for train/test sets
X_train = df2_scaled_train[['standardscaler__grade_score']]
y_train = to_categorical(df2.loc[df2_scaled_train.index, 'loan_status_code'].astype(int))
X_test = df2_scaled_test[['standardscaler__grade_score']]
y_test = to_categorical(df2.loc[df2_scaled_test.index, 'loan_status_code'].astype(int))

# Train the model
model = Sequential([
    Dense(31, activation='relu'),
    Dense(8, activation='softmax')
])
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
history2 = model.fit(X_train, y_train, epochs=epochs, validation_data=(X_test, y_test))

# Plot the results over time
plt.plot(history2.history['loss'])
plt.plot(history2.history['val_loss'])
plt.legend(['Training loss', 'Validation loss'])

plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')

Aha, it looks like this one definitely performed worse… so there _is_ an unfair advantage to the random data splitting of `train_test_split`; so for the remainder of this analysis, we'll rely on the fairer and more realistic time-based splitting.

We're off to a good start. Let's see if we can perform better by also considering the loan amount.

In [ ]:
df3 = accepted_df[['grade_score', 'loan_amnt', 'loan_status_code', 'issue_d_month_index']].dropna()

df3 = df3.sample(frac=data_frac)

t = make_column_transformer((StandardScaler(), ['grade_score', 'loan_amnt']),
                            remainder='passthrough').set_output(transform="pandas")
df3_scaled = t.fit_transform(df3)

X = df3_scaled[['standardscaler__grade_score', 'standardscaler__loan_amnt']]
y = to_categorical(df3_scaled['remainder__loan_status_code'].astype(int))

## Define X and y, split based on time. Reuse cutoff_index from above.
df3_scaled_train = df3_scaled[df3['issue_d_month_index'] <= cutoff_index]
df3_scaled_test = df3_scaled[df3['issue_d_month_index'] > cutoff_index]
X_train = df3_scaled_train[['standardscaler__grade_score', 'standardscaler__loan_amnt']]
X_test = df3_scaled_test[['standardscaler__grade_score', 'standardscaler__loan_amnt']]
y_train = to_categorical(df3_scaled_train['remainder__loan_status_code'].astype(int))
y_test = to_categorical(df3_scaled_test['remainder__loan_status_code'].astype(int))

# Train the model
model = Sequential([
    Dense(31, activation='relu'),
    Dense(8, activation='softmax')
])
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
history3 = model.fit(X_train, y_train, epochs=epochs, validation_data=(X_test, y_test))

# Plot the results over time
plt.plot(history3.history['loss'])
plt.plot(history3.history['val_loss'])
plt.legend(['Training loss', 'Validation loss'])

plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')

Even better! Loan issuers will certainly be considering loan amounts in addition to general borrower grades, and this shows that loan amounts do factor in to the actual outcome. Also, since validation loss continues to drop as training continues, this suggests that we are not yet overfitting.

Now let's see if we can make better predictions by factoring time into this.

In [ ]:
df4 = accepted_df[['grade_score', 'loan_amnt', 'issue_d_month_index', 'loan_status_code']].dropna()
df4 = df4.sample(frac=data_frac)

t = make_column_transformer((StandardScaler(), ['grade_score', 'loan_amnt', 'issue_d_month_index']),
                            remainder='passthrough').set_output(transform="pandas")
df4_scaled = t.fit_transform(df4)

## Define X and y, split based on time. Reuse cutoff_index from above.
df4_scaled_train = df4_scaled[df4['issue_d_month_index'] <= cutoff_index]
df4_scaled_test = df4_scaled[df4['issue_d_month_index'] > cutoff_index]
X_train = df4_scaled_train[['standardscaler__grade_score', 'standardscaler__loan_amnt', 'standardscaler__issue_d_month_index']]
X_test = df4_scaled_test[['standardscaler__grade_score', 'standardscaler__loan_amnt', 'standardscaler__issue_d_month_index']]
y_train = to_categorical(df4_scaled_train['remainder__loan_status_code'].astype(int))
y_test = to_categorical(df4_scaled_test['remainder__loan_status_code'].astype(int))

# Train the model
model = Sequential([
    Dense(31, activation='relu'),
    Dense(8, activation='softmax')
])
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
history4 = model.fit(X_train, y_train, epochs=epochs, validation_data=(X_test, y_test))

# Plot the results over time
plt.plot(history4.history['loss'])
plt.plot(history4.history['val_loss'])
plt.legend(['Training loss', 'Validation loss'])

plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')

So, as it turns out, we can see a dramatic improvement in accuracy and loss by factoring in loan origination time along with other borrower data. And once again, since validation loss continues to drop, we are probably not overfitting.

Of course, this is "cheating" in the sense that it takes into account information about time, which will correlate with eventual outcomes, at a time when those future outcomes are not yet known. And yet, this provides some level of assurance that we can look at prevailing macroeconomic conditions that _are_ known at the time, and use them to make better predictions.

Let's explore the hypothesis that this actually correlates with economic conditions. To do this, let's take a measure of stock market performance (_without_ the federal funds rate), and see how that compares with the best analysis from above, incorporating information about both borrower grade and loan amount, but _not_ time.

In [ ]:
df5 = accepted_df.rename(columns={'issue_d_month_index':'date_index'})

# Merge in S&P 500 columns
df5 = df5.merge(
    sp500_df[['date_index', 'Close', 'avg_close', 'close_above_avg']],
    on='date_index',
    how='left'
).dropna()

df5 = df5.sample(frac=data_frac)

t = make_column_transformer((StandardScaler(), ['grade_score', 'loan_amnt', 'close_above_avg']),
                            remainder='passthrough').set_output(transform="pandas")
df5_scaled = t.fit_transform(df5)

## Define X and y, split based on time. Reuse cutoff_index from above.
df5_scaled_train = df5_scaled[df5['date_index'] <= cutoff_index]
df5_scaled_test = df5_scaled[df5['date_index'] > cutoff_index]
X_train = df5_scaled_train[['standardscaler__grade_score', 'standardscaler__loan_amnt', 'standardscaler__close_above_avg']]
X_test = df5_scaled_test[['standardscaler__grade_score', 'standardscaler__loan_amnt', 'standardscaler__close_above_avg']]
y_train = to_categorical(df5_scaled_train['remainder__loan_status_code'].astype(int))
y_test = to_categorical(df5_scaled_test['remainder__loan_status_code'].astype(int))

# Train the model
model = Sequential([
    Dense(31, activation='relu'),
    Dense(8, activation='softmax')
])
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
history5 = model.fit(X_train, y_train, epochs=epochs, validation_data=(X_test, y_test))

# Plot the results over time, for both history3 (the best above) and history 5 (with the S&P 500)
plt.plot(history3.history['loss'], label='Baseline loss')
plt.plot(history3.history['val_loss'], label='Baseline validation loss')

plt.plot(history5.history['loss'], label='S&P loss')
plt.plot(history5.history['val_loss'], label='S&P validation loss')
plt.legend(['Baseline Loss', 'Baseline validation loss', 'S&P loss', 'S&P validation loss'])

plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')

Wow! The model improves dramatically when we factor in information from the S&P 500. Let's see if it improves with _only_ the federal funds rate instead of the S&P 500. And finally, let's see if it improves if we include _both_.

In [ ]:
df6 = accepted_df.rename(columns={'issue_d_month_index':'date_index'})

# Merge in federal funds rate data
df6 = df6.merge(
    fed_funds_df[['date_index', 'FEDFUNDS', 'dFEDFUNDS']],
    on='date_index',
    how='left'
).dropna()

df6 = df6.sample(frac=data_frac)

t = make_column_transformer((StandardScaler(), ['grade_score', 'loan_amnt', 'FEDFUNDS', 'dFEDFUNDS']),
                            remainder='passthrough').set_output(transform="pandas")
df6_scaled = t.fit_transform(df6)

# Split data based on time cutoffs
df6_scaled_train = df6_scaled[df6['date_index'] <= cutoff_index]
df6_scaled_test = df6_scaled[df6['date_index'] > cutoff_index]
X_train = df6_scaled_train[['standardscaler__grade_score', 'standardscaler__loan_amnt', 'standardscaler__FEDFUNDS', 'standardscaler__dFEDFUNDS']]
X_test = df6_scaled_test[['standardscaler__grade_score', 'standardscaler__loan_amnt', 'standardscaler__FEDFUNDS', 'standardscaler__dFEDFUNDS']]
y_train = to_categorical(df6_scaled_train['remainder__loan_status_code'].astype(int))
y_test = to_categorical(df6_scaled_test['remainder__loan_status_code'].astype(int))

# Train the model
model = Sequential([
    Dense(31, activation='relu'),
    Dense(8, activation='softmax')
])
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
history6 = model.fit(X_train, y_train, epochs=epochs, validation_data=(X_test, y_test))



In [ ]:
df7 = accepted_df.rename(columns={'issue_d_month_index':'date_index'})

# Merge in S&P 500 columns
df7 = df7.merge(
    sp500_df[['date_index', 'Close', 'avg_close', 'close_above_avg']],
    on='date_index',
    how='left'
).dropna()

# Merge in federal funds rate data
df7 = df7.merge(
    fed_funds_df[['date_index', 'FEDFUNDS', 'dFEDFUNDS']],
    on='date_index',
    how='left'
)

df7 = df7.sample(frac=data_frac)

t = make_column_transformer((StandardScaler(), ['grade_score', 'loan_amnt', 'FEDFUNDS', 'dFEDFUNDS', 'close_above_avg']),
                            remainder='passthrough').set_output(transform="pandas")
df7_scaled = t.fit_transform(df7)

# Split the data based on time cutoffs
df7_scaled_train = df7_scaled[df7['date_index'] <= cutoff_index]
df7_scaled_test = df7_scaled[df7['date_index'] > cutoff_index]
X_train = df7_scaled_train[['standardscaler__grade_score', 'standardscaler__loan_amnt', 'standardscaler__FEDFUNDS', 'standardscaler__dFEDFUNDS', 'standardscaler__close_above_avg']]
X_test = df7_scaled_test[['standardscaler__grade_score', 'standardscaler__loan_amnt', 'standardscaler__FEDFUNDS', 'standardscaler__dFEDFUNDS', 'standardscaler__close_above_avg']]
y_train = to_categorical(df7_scaled_train['remainder__loan_status_code'].astype(int))
y_test = to_categorical(df7_scaled_test['remainder__loan_status_code'].astype(int))

# Train the model
model = Sequential([
    Dense(31, activation='relu'),
    Dense(31, activation='sigmoid'),
    Dense(31, activation='sigmoid'),
    Dense(8, activation='softmax')
])
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
history7 = model.fit(X_train, y_train, epochs=epochs, validation_data=(X_test, y_test))



In [ ]:
# Plot the results over time -- for S&P 500 only, fed funds only, and both combined
plt.plot(history5.history['loss'], label='S&P loss')
plt.plot(history5.history['val_loss'], label='S&P validation loss')
plt.plot(history6.history['loss'], label='FF loss')
plt.plot(history6.history['val_loss'], label='FF validation loss')
plt.plot(history7.history['loss'], label='S&P+FF loss')
plt.plot(history7.history['val_loss'], label='S&P+FF validation loss')
plt.legend(['S&P loss', 'S&P validation loss', 'FF loss', 'FF validation loss', 'S&P+FF loss', 'S&P+FF validation loss'])

plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.show()

Finally we've found our best model. We already established above that using _only_ the S&P 500 data, in addition to traditional loan metrics, performs better than the baseline.

Here we've shown that using _only_ the Effective Federal Funds Rate performs better than using _only_ the S&P 500.

And we've also shown that using _both_ the Effective Federal Funds Rate _and_ the S&P 500 performs better than _either_ alone.

Now, to summarize, let's visualize the validation loss of the best model, including all available data that was known at loan origination time, and compare that with the baseline that excludes the fed funds rate. Let's also visualize this with validation accuracy.

In [ ]:
# Plot the results over time, directly comparing baseline with best model
plt.plot(history3.history['val_loss'], label='Baseline validation loss')
plt.plot(history7.history['val_loss'], label='Validation loss with econ data')
plt.legend(['Baseline validation loss', 'Validation loss including econ data'])

plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')

In [ ]:
# Create a figure with two subplots side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Validation Loss ---
ax1.plot(history3.history['val_loss'], label='Baseline', color='blue')
ax1.plot(history7.history['val_loss'], label='Including econ', color='orange')
ax1.set_title('Model Validation Loss')
ax1.set_ylabel('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Plot 2: Validation Accuracy ---
# Note: Use 'val_accuracy' if 'val_acc' results in a KeyError
ax2.plot(history3.history['val_accuracy'], label='Baseline', color='blue')
ax2.plot(history7.history['val_accuracy'], label='Including econ', color='orange')
ax2.set_title('Model Validation Accuracy')
ax2.set_ylabel('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Adjust layout to prevent overlapping labels
plt.tight_layout()
plt.show()

# Summary
This analysis examined actual loan payback performance over time, in an effort to see if predictions could be improved.

## Methodology
We imported the Lending Club dataset from Kaggle, along with S&P 500 closing prices and Effective Federal Funds Rate data.

Loan data was analyzed using simply the borrower grade as determined by Lending Club (trusting their analysis to be a financially-motivated estimate and therefore a reasonably accurate one according to historical measures).

S&P 500 data was used in the analysis based only on the closing price above recent averages. Thus, we examined when the market was performing well compared with recent history.

Effective Federal Funds Rate data was examined using both its absolute level as well as how much it changed each day. Thus we examined the overall federal funds rate, in addition to showing measures when it was increasing or decreasing.

All analyses were conducted using a 2-layer neural net.

We performed the following analyses:

| Analysis | Loan data | S&P 500 Data | Fed Data |
|----------|-----------|--------------|----------|
| Baseline | Yes       | No           | No       |
| S&P Only | Yes       | Yes          | No       |
| Fed Only | Yes       | No           | Yes      |
| All Data | Yes       | Yes          | Yes      |

## Results
All analyses beat the baseline by a significant margin, thus establishing that macroeconomic data **can** be a significant additional signal for predicting overall loan performance.

S&P 500 data alone was helpful. Fed data alone was even more helpful. Unexpectedly, when these two data sources were combined, they performed _worse_ than the Fed data alone. This condition tended to persist even with various experiments in model architecture and training time (not detailed here).

This is particularly surprising since, if a particular data column does not improve results, we would expect the model to learn to ignore it. The author's best explanation here is that stock market data possesses sufficient inherent randomness that it conveys confounding noise that is nonetheless useful often enough to be retained by the model; however, the smooth movement of the Fed funds rate, set manually by policymakers, tends to be a better reflection of macroeconomic conditions.

## Further Work

Incorporating more macroeconomic signals, and modeling them better, would likely result in further improvements to model performance.